# Course project

This notebook includes the code for the course project in the DTU course [Integrated Energy Grids](https://kurser.dtu.dk/course/2024-2025/46770?menulanguage=en)  and is modelling and analyzing the german energy system. 
 
The report is provided along this notebook, further explaining the results. 

### Imports

In [ ]:
import pandas as pd
import pypsa
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import os
import sys

# Set path to the directory containing base.py
module_path = os.path.abspath(".")

if module_path not in sys.path:
    sys.path.append(module_path)

import base  # now you can use base.build_network()

from visualization import *

In [ ]:
costs = base.load_technology_data()
country = "DEU"

## E. Decarbonization
Select one target for decarbonization (i.e., one CO2 allowance limit). What is the CO2 price required to achieve that decarbonization level? Search for information on the existing CO2 tax in your country (if any) and discuss your results.

In [ ]:
network = base.build_network(solve=False)

In [ ]:
#Create a new bus
network.add("Bus",
          "elec_store",
          carrier = "battery storage",
          overwrite=True)

#Connect the store to the bus
network.add("Store",
          "Battery Storage Store",
          bus = "elec_store",
          e_nom_extendable = True,
          e_cyclic = True,
          capital_cost = costs.at["battery storage", "capital_cost"],
          overwrite=True)

network.add("Link",
          "grid to battergy storage",
          bus0 = f"{country}_elec",
          bus1 = "elec_store",
          p_nom_extendable = True,
          efficiency = costs.at["battery inverter", "efficiency"],
          capital_cost = costs.at["battery inverter", "efficiency"],
          overwrite=True)

network.add("Link",
          "battery storage to grid",
          bus0 = "elec_store",
          bus1 = f"{country}_elec",
          p_nom_extendable = True,
          efficiency = costs.at["battery inverter", "efficiency"],
          capital_cost = costs.at["battery inverter", "efficiency"],
          overwrite=True
          )

#Create a new carrier
network.add("Carrier",
              "H2")

#Create a new bus
network.add("Bus",
          "H2",
          carrier = "H2",
          overwrite=True)

#Connect the store to the bus
network.add("Store",
          "H2 Tank",
          bus = "H2",
          e_nom_extendable = True,
          e_cyclic = True,
          capital_cost = base.annuity(25, 0.07)*57000*(1+0.011),
          overwrite=True)

network.add("Link",
          "H2 Electrolysis",
          bus0 = f"{country}_elec",
          bus1 = "H2",
          p_nom_extendable = True,
          efficiency = 0.8,
          capital_cost = base.annuity(25, 0.07)*600000*(1+0.05),
          overwrite=True)

#Add the link "H2 Fuel Cell" that transports energy from the H2 bus (bus0) to the electricity bus (bus1)
#with 58% efficiency
network.add("Link",
          "H2 Fuel Cell",
          bus0 = "H2",
          bus1 = f"{country}_elec",
          p_nom_extendable = True,
          efficiency = 0.58,
          capital_cost = base.annuity(10, 0.07)*1300000*(1+0.05),
          overwrite=True)

Add CO2 constraint of german emissions allowance for 2040, 88 reduction compared to 1990

In [ ]:
co2_emissions_2040 = 0.12*366e6
network.add("GlobalConstraint", "CO2Limit",
          carrier_attribute="co2_emissions",
          sense="<=",
          constant=co2_emissions_2040)
print("co2 allowance: ", co2_emissions_2040/10e6, "tCo2")

In [ ]:
network.optimize(solver_name="gurobi", solver_options={"InfUnbdInfo": 1})

In [ ]:
# check some numbers

print(f"System cost: {round(network.objective / 1e9, 2)} billion euros")
print("")
print("Optimal Generator Capactities in GW:")
print(network.generators.p_nom_opt.div(1e3))  # MW -> GW
print("")
print("Optimal Energy Generation in GWh/a")
print(network.generators_t.p.sum().div(1e6))

# Total emissions in the system (sum over all generators and time)
actual_emissions = (
    network.generators_t.p
    .multiply(network.generators.carrier.map(network.carriers["co2_emissions"]))
    .sum()
    .sum()
)

# battery setup
print("")
print("Battery Storage results:")
print("Optimal Storage Size (Energy Content): ", round(network.stores.loc["Battery Storage Store"]["e_nom_opt"]/10e3, 2), "GWh")
print("Optimal nominal output power battery: ", round(network.links.loc["battery storage to grid"]["p_nom_opt"]/10e3,2), "GW")
print("Optimal nominal input power battery: ", round(network.links.loc["grid to battergy storage"]["p_nom_opt"]/10e3,2), "GW")

print("")
# The imposed CO₂ limit (from GlobalConstraint)
co2_limit = network.global_constraints.at["CO2Limit", "constant"]

print(f"Actual emissions: {actual_emissions:.2e} t")
print(f"CO₂ constraint  : {co2_limit:.2e} t")

dual = network.global_constraints.at["CO2Limit", "mu"]
print(f"Shadow price of CO₂: {dual:.2f} €/tCO2")

In [ ]:
(network.generators_t.p.sum().div(1e6).sum()-(20.92))/network.generators_t.p.sum().div(1e6).sum()